# 주제 2 baseline — 보행자 분할 분석 (Penn-Fudan)

부경대학교 교내 컴퓨터비전 부트캠프 · 최종 프로젝트 baseline · 2026. 8. 7.

---

정답 마스크가 깔끔하게 갖춰진 데이터로 **prompt 방식별 SAM 성능**을 비교합니다. 해커톤에서 했던 '한 번에 하나씩 바꾸기' 를 그대로 적용할 수 있는 주제입니다.

| STEP | 하는 일 |
|---|---|
| 0 · 1 | 환경 준비 · Penn-Fudan 내려받기 (약 53 MB) |
| 2 | SAM 2.1 불러오기 |
| 3 | prompt 방식 4가지로 분할 |
| 4 | 결과를 표로 모으기 |
| 5 | **정량 분석** — 방식별 IoU 비교 |
| 6 | **실패 사례** 고르고 저장 |

> 반려동물(Oxford-IIIT Pet)로 하고 싶다면 STEP 1 의 대체 셀을 쓰세요.

**시작 전에** — `런타임 → 런타임 유형 변경 → T4 GPU`. 그리고 STEP 1 의 다운로드 셀을
가장 먼저 실행해 두세요. 받는 동안 아래를 읽으면 됩니다.

## STEP 0 · 환경 준비

In [ ]:
!pip install -q ultralytics

In [ ]:
# 그래프 한글 폰트 (실패해도 실습에는 지장 없음)
try:
    !apt-get install -qq -y fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    import matplotlib.pyplot as plt
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 준비 완료")
except Exception as e:
    print("폰트 설치 건너뜀:", e)

In [ ]:
import os, glob, json, urllib.request
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
np.random.seed(0)


def show(img_bgr, title=None, w=9):
    h = w * img_bgr.shape[0] / img_bgr.shape[1]
    plt.figure(figsize=(w, h))
    plt.imshow(img_bgr[:, :, ::-1]); plt.axis("off")
    if title:
        plt.title(title, fontsize=12)
    plt.show()


def overlay(img_bgr, mask, color=(60, 120, 240), alpha=0.5):
    layer = np.zeros_like(img_bgr)
    layer[np.asarray(mask).astype(bool)] = color
    return cv2.addWeighted(img_bgr, 1.0, layer, alpha, 0)


def mask_iou(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix = max(0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def qbin(series, q, names):
    """값을 분위 구간으로 나눈다. 값이 적어 구간이 줄어도 죽지 않는다."""
    b = pd.qcut(series, q, duplicates="drop")
    cats = list(b.cat.categories)
    lab = names[:len(cats)] if len(cats) <= len(names) else [str(c) for c in cats]
    return b.cat.rename_categories(lab)


print("준비 완료")

In [ ]:
os.makedirs("outputs", exist_ok=True)     # 결과 이미지를 여기에 저장한다
os.makedirs("data", exist_ok=True)
print(os.listdir("."))

## STEP 1 · 데이터 준비 — Penn-Fudan Pedestrian

보행자 170장, **사람마다 픽셀 단위 정답 마스크**가 들어 있습니다.
PyTorch 공식 튜토리얼에서도 쓰는 데이터라 구조가 단순합니다.

In [ ]:
if not os.path.isdir("PennFudanPed"):
    !wget -q https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip
    !unzip -q -o PennFudanPed.zip
print(sorted(os.listdir("PennFudanPed"))[:6])

In [ ]:
N_IMAGES = 40      # 처음에는 40장으로 시작 (전체 170장)

img_files = sorted(glob.glob("PennFudanPed/PNGImages/*.png"))[:N_IMAGES]
msk_files = [f.replace("PNGImages", "PedMasks").replace(".png", "_mask.png")
             for f in img_files]
print("이미지", len(img_files), "장")
print(img_files[0])
print(msk_files[0], os.path.exists(msk_files[0]))

정답 마스크는 사람마다 1, 2, 3 … 번호가 칠해진 PNG 입니다.
번호별로 떼어 내면 인스턴스 마스크가 됩니다.

In [ ]:
def load_gt(mask_path):
    """정답 마스크 PNG → [(사람 마스크, 박스), ...]"""
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    out = []
    for v in np.unique(m):
        if v == 0:
            continue
        mm = (m == v)
        ys, xs = np.where(mm)
        out.append((mm, [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]))
    return out


img0 = cv2.imread(img_files[0])
gt0 = load_gt(msk_files[0])
print("첫 이미지의 사람 수:", len(gt0))

vis = img0.copy()
for mm, bx in gt0:
    vis = overlay(vis, mm, (50, 180, 90), 0.45)
    cv2.rectangle(vis, (bx[0], bx[1]), (bx[2], bx[3]), (50, 180, 90), 2)
show(vis, "정답 마스크(초록) — 이것과 비교한다")

## STEP 2 · SAM 불러오기

In [ ]:
from ultralytics import SAM

sam = SAM("sam2.1_b.pt")
print("SAM 준비 완료")

## STEP 3 · prompt 방식 네 가지

같은 사람에 대해 prompt 만 바꿔 가며 마스크를 만듭니다.
**한 번에 하나씩만 바꾸는** 전형적인 비교 실험입니다.

| 방식 | prompt |
|---|---|
| `box` | 정답 박스를 그대로 |
| `point1` | 박스 중심에 점 하나 |
| `point3` | 세로로 세 점 (머리·몸통·다리) |
| `box_center` | 박스 + 중심점 |

In [ ]:
def prompts_of(box, mode):
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    h = box[3] - box[1]
    if mode == "box":
        return dict(bboxes=[box])
    if mode == "point1":
        return dict(points=[[cx, cy]], labels=[1])
    if mode == "point3":
        pts = [[cx, box[1] + h * 0.15], [cx, cy], [cx, box[3] - h * 0.15]]
        return dict(points=[pts], labels=[[1, 1, 1]])
    if mode == "box_center":
        return dict(bboxes=[box], points=[[cx, cy]], labels=[1])
    raise ValueError(mode)


MODES = ["box", "point1", "point3", "box_center"]
print(MODES)

In [ ]:
# 40장 × 사람 수 × 4방식 — T4 에서 3~6분 걸립니다
records, store = [], {}
for k, (ip, mp) in enumerate(zip(img_files, msk_files)):
    gts = load_gt(mp)
    for pi, (gmask, gbox) in enumerate(gts):
        for mode in MODES:
            try:
                r = sam(ip, verbose=False, **prompts_of(gbox, mode))[0]
                pred = r.masks.data[0].cpu().numpy()
            except Exception:
                pred = np.zeros_like(gmask)
            records.append(dict(image=os.path.basename(ip), person=pi, mode=mode,
                                iou=round(mask_iou(pred, gmask), 4),
                                gt_area=int(gmask.sum())))
            store[(ip, pi, mode)] = pred
    if (k + 1) % 10 == 0:
        print(k + 1, "/", len(img_files), flush=True)

df = pd.DataFrame(records)
print(df.shape)
df.head()

## STEP 4 · 결과 표

In [ ]:
pivot = df.pivot_table(index="mode", values="iou", aggfunc=["mean", "median", "count"])
pivot.columns = ["평균 IoU", "중앙값", "건수"]
print(pivot.round(3).sort_values("평균 IoU", ascending=False))

## STEP 5 · 정량 분석 ★

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = df.groupby("mode").iou.mean().sort_values().index
axes[0].boxplot([df[df["mode"] == m].iou for m in order])
axes[0].set_xticklabels(list(order))
axes[0].set_ylabel("mask IoU"); axes[0].set_title("prompt 방식별 IoU 분포", fontsize=12)
axes[0].axhline(0.5, color="#E97132", ls="--", lw=1.6)

for m in order:
    axes[1].hist(df[df["mode"] == m].iou, bins=20, histtype="step", lw=2, label=m)
axes[1].legend(); axes[1].set_xlabel("mask IoU"); axes[1].set_title("겹쳐 보기", fontsize=12)
plt.tight_layout(); plt.show()

print("0.5 미만 비율")
print((df.groupby("mode").iou.apply(lambda s: (s < 0.5).mean() * 100)).round(1))

### 여기에 여러분의 질문을 하나 더

- **작은 사람**(gt_area 하위 25%)에서는 순위가 달라지는가?
- 한 이미지에 **사람이 많을수록** IoU 가 떨어지는가?
- `point1` 이 실패한 건들은 어디를 찍었기에 실패했는가?

In [ ]:
# 예시 — 크기 구간별로 방식 순위가 바뀌는지 본다
df["size_bin"] = qbin(df.gt_area, 3, ["작음", "중간", "큼"])
tbl = df.pivot_table(index="size_bin", columns="mode", values="iou", aggfunc="mean")
print(tbl.round(3))

tbl.plot(kind="bar", figsize=(7.5, 4))
plt.ylabel("평균 mask IoU"); plt.title("크기 구간 × prompt 방식", fontsize=12)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## STEP 6 · 실패 사례 ★

In [ ]:
BASE = "box"       # 기준으로 삼을 방식
sub = df[df["mode"] == BASE].sort_values("iou")

def draw_pair(ip, pi, mode, tag, iou):
    img = cv2.imread(ip)
    gts = load_gt(ip.replace("PNGImages", "PedMasks").replace(".png", "_mask.png"))
    gmask = gts[pi][0]
    a = overlay(img, gmask, (50, 180, 90), 0.45)          # 정답 = 초록
    b = overlay(img, store[(ip, pi, mode)], (232, 96, 21), 0.45)  # 예측 = 주황
    canvas = np.concatenate([a, b], axis=1)
    cv2.imwrite(f"outputs/{tag}.png", canvas)
    show(canvas, f"{tag}  —  왼쪽 정답 / 오른쪽 예측 ({mode}, IoU {iou:.2f})", w=13)

for i, r in enumerate(sub.head(3).itertuples(), 1):
    ip = [p for p in img_files if os.path.basename(p) == r.image][0]
    draw_pair(ip, r.person, BASE, f"failure_{i}", r.iou)
for i, r in enumerate(sub.tail(3).itertuples(), 1):
    ip = [p for p in img_files if os.path.basename(p) == r.image][0]
    draw_pair(ip, r.person, BASE, f"success_{i}", r.iou)
print("저장:", sorted(os.listdir("outputs")))

### 실패 사례 캡션 (직접 채우세요)

| 파일 | IoU | 무엇이 문제였나 (가림 · 작음 · 그림자 · 여러 명 겹침 …) |
|---|---|---|
| failure_1.png | | |
| failure_2.png | | |
| failure_3.png | | |

세 장의 **공통점**을 한 문장으로: 

## STEP 7 · 결과 이미지 내려받기

Colab 에서 `outputs/` 폴더를 통째로 압축해 내려받습니다.

In [ ]:
!zip -q -r outputs.zip outputs
try:
    from google.colab import files
    files.download("outputs.zip")
except Exception as e:
    print("Colab 이 아닙니다:", e)
print("저장된 이미지:", len(glob.glob("outputs/*.png")), "장")

## STEP 8 · 심화 트랙 (선택)

여기까지 여유 있게 끝냈다면 **`5일차_심화_finetuning_가이드.ipynb`** 를 여세요.
사전학습 모델을 그대로 쓰는 대신 **직접 fine-tuning 해서 같은 잣대로 비교**하는 트랙입니다.
평가에 **심화 10점**이 따로 배정되어 있습니다.

- 먼저 **위의 STEP 6 까지를 끝내 두어야** 합니다 — 기준선이 없으면 비교가 성립하지 않습니다
- T4 GPU 기준 30 epoch 에 10 ~ 15분
- **11:20 까지 여기에 도달하지 못했다면 심화는 포기하고 발표 준비로 넘어가세요**

심화는 fine-tuning 만 인정하는 것이 아닙니다. 새 지표를 직접 만들었거나, 데이터를 늘려
다시 쟀거나, prompt 전략을 체계적으로 비교했어도 같은 점수입니다.

---

## 제출 전 점검

- [ ] **런타임 → 런타임 다시 시작** 후 처음부터 끝까지 오류 없이 실행되는가
- [ ] `outputs/` 폴더에 성공 사례 3장 이상, 실패 사례 3장 이상
- [ ] 결과 이미지마다 캡션(파일명 또는 아래 마크다운 셀)이 붙어 있는가
- [ ] 표 또는 그래프가 하나 이상 있는가
- [ ] 아래 요약 다섯 줄을 채웠는가

## 결과 요약 (이 셀을 더블클릭해 직접 채우세요)

1. **무엇을 만들었나** —
2. **정량 결과** — (숫자 하나 이상)
3. **가장 잘 된 경우** —
4. **실패 유형과 개수** —
5. **시간이 더 있었다면** —
